# **Binary Model**

## Libraries

In [19]:
# Libraries
import sys
import pandas as pd
from tensorflow import keras
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score

from tensorflow.keras.preprocessing.image import load_img, img_to_array, smart_resize, ImageDataGenerator

from tensorflow.keras.applications.efficientnet import EfficientNetB0
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import optuna

sys.path.append('../src')  # Add root folder of project

from display_utils import show_image
from _constants import DATA_DIR, IMAGE_DIR, IMAGE_SIZE_STANDARD
from preprocess_utils import resize_image

/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# Setting options
pd.set_option('display.max_rows', None)

## Data Loading

In [21]:
# Load the metadata
df = pd.read_csv(f'{DATA_DIR}/metadata.csv')

# Load the IsAnimal data
label = pd.read_csv(f'{DATA_DIR}/binary_labels.csv')

# Join the two dataframes
data = pd.merge(
    df
    ,label
    ,left_on='rare_species_id'
    ,right_on='image_id'
    ,how='inner'
).drop(columns=['rare_species_id', 'image_id'])

In [22]:
# # Creating a sample
# train_df, sample_df = train_test_split(
#     data,
#     test_size=0.3,
#     stratify=data['family'],
#     random_state=20
# )

## Data Preprocessing

In [23]:
# Performing the splits
train_df, test_df = train_test_split(data, test_size=0.2, stratify=data['family'], random_state=20)  # Create test set
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['family'], random_state=20)  # Create validation set

In [24]:
# Encoding the target
enc = LabelEncoder()

train_df['family'] = enc.fit_transform(train_df['family'])
val_df['family'] = enc.transform(val_df['family'])
test_df['family'] = enc.transform(test_df['family'])

In [25]:
# Image resizing target dimensions
IMG_SIZE = 224
BATCH_SIZE = 8

In [26]:
# Defining functions to resize the images
# def smart_resize_img(file_path, target_size=(IMG_SIZE, IMG_SIZE)):
#     img = load_img(f'{IMAGE_DIR}/{file_path}')  # Load image
#     img_array = img_to_array(img)  # Convert to array
#     resized_img = smart_resize(img_array, target_size)  # Resize image
#     resized_img /= 255.0
    
#     return resized_img

def smart_resize_img(image, target_size=(IMG_SIZE, IMG_SIZE)):
    resized_img = smart_resize(image, target_size)  # Resize image
    resized_img /= 255.0  # Normalize the image
    
    return resized_img

# Custom generator that uses smart_resize
# def smart_resize_generator(dataframe, target, batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE)):
#     while True:
#         # Iterate through the dataframe in batches
#         for i in range(0, len(dataframe), batch_size):
#             batch_df = dataframe.iloc[i:i+batch_size]
            
#             # Prepare the batch of images and labels
#             batch_images = np.array([smart_resize_img(file_path, target_size) for file_path in batch_df['file_path']])
#             batch_labels = np.array(batch_df[target])
            
#             yield batch_images, batch_labels

In [27]:
# # Resizing the images
# label_train_generator = smart_resize_generator(train_df, 'label', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
# print('train resizing completed')
# label_val_generator = smart_resize_generator(val_df, 'label', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
# print('validation resizing completed')
# label_test_generator = smart_resize_generator(test_df, 'label', batch_size=BATCH_SIZE, target_size=(IMG_SIZE, IMG_SIZE))
# print('test resizing completed')

In [28]:
datagen = ImageDataGenerator(preprocessing_function=smart_resize_img)

# Train generator
binary_train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=f'{IMAGE_DIR}',
    x_col='file_path',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=True  # shuffle for training
)

# Validation generator
binary_val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=f'{IMAGE_DIR}',
    x_col='file_path',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=False
)

# Test generator
binary_test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=f'{IMAGE_DIR}',
    x_col='file_path',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=False
)

Found 8148 validated image filenames.
Found 1438 validated image filenames.
Found 2397 validated image filenames.


## Modelling

In [33]:
def objective(trial):
    """
    Objective function for Optuna hyperparameter optimization.
    
    Args:
    - trial (optuna.trial.Trial): The trial object used to sample hyperparameters.
    
    Returns:
    - float: The evaluation metric (validation accuracy or other).
    """
    lr = trial.suggest_float("lr", 1e-6, 1e-1)
    optimizer_type = trial.suggest_categorical("optimizer", ["adam", "sgd"])
    patience = trial.suggest_int("patience", 1, 3)
    lr_factor = trial.suggest_float("lr_factor", 0.1, 0.7)
    lr_patience = trial.suggest_int("lr_patience", 1, 5)

    # Set the input
    input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Load the pre-trained model
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_tensor)

    # Add necessary layers
    x = GlobalAveragePooling2D()(base_model.output)
    output = Dense(1, activation='sigmoid')(x)  # Binary classification

    binary_model = Model(inputs=input_tensor, outputs=output)

    # Optimizer setup based on dynamic 'optimizer_type'
    if optimizer_type == "adam":
        optimizer = Adam(learning_rate=lr)
    elif optimizer_type == "sgd":
        optimizer = SGD(learning_rate=lr)
    else:
        raise ValueError(f"Unknown optimizer type: {optimizer_type}")
    
    # Compiling the model
    binary_model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    # Fit the model
    binary_model.fit(
        binary_train_generator,
        validation_data=binary_val_generator,
        epochs=2,
        steps_per_epoch=int(np.ceil(len(train_df) / BATCH_SIZE)),
        callbacks=[
            EarlyStopping(patience=patience, restore_best_weights=True),
            ReduceLROnPlateau(patience=lr_patience, factor=lr_factor, verbose=1)
        ],
        verbose=1
    )

    # Evaluate the model using the  validation set
    val_loss, val_accuracy = binary_model.evaluate(binary_val_generator, verbose=0)
    
    # Return the validation accuracy as the objective metric
    return val_accuracy

In [34]:
# Create a study
study = optuna.create_study(direction="maximize")

# Optimize the objective function
study.optimize(lambda trial: objective(trial), n_trials=2)

# Get the best trial's hyperparameters
print(f"Best trial: {study.best_trial.params}")

[I 2025-04-15 22:17:13,205] A new study created in memory with name: no-name-c6d47fd2-761a-4766-8252-d762926817a7


Epoch 1/2
 175/1019 ━━━━━━━━━━━━━━━━━━━━ 11:43 833ms/step - accuracy: 0.8724 - loss: 0.3567 - precision: 0.9254 - recall: 0.9389

/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (115600000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


 569/1019 ━━━━━━━━━━━━━━━━━━━━ 6:07 816ms/step - accuracy: 0.9108 - loss: 0.2667 - precision: 0.9341 - recall: 0.9727

[W 2025-04-15 22:25:13,106] Trial 0 failed with parameters: {'lr': 0.01053391386829343, 'optimizer': 'sgd', 'patience': 1, 'lr_factor': 0.5442611346936899, 'lr_patience': 2} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_14398/1601219436.py", line 5, in <lambda>
    study.optimize(lambda trial: objective(trial), n_trials=2)
                                 ^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_14398/6580981.py", line 45, in objective
    binary_model.fit(
  File "/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/utils/traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/backend/

KeyboardInterrupt: 

In [12]:
# Build the model

# Set the input
input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Load the pre-trained model
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=input_tensor)

# Add necessary layers
x = GlobalAveragePooling2D()(base_model.output)
output = Dense(1, activation='sigmoid')(x)  # Binary classification

binary_model = Model(inputs=input_tensor, outputs=output)

2025-04-15 19:55:23.642730: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_UNKNOWN: unknown error
2025-04-15 19:55:23.642767: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:135] retrieving CUDA diagnostic information for host: shadybea
2025-04-15 19:55:23.642772: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:142] hostname: shadybea
2025-04-15 19:55:23.642940: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:166] libcuda reported version is: 550.120.0
2025-04-15 19:55:23.642965: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] kernel reported version is: 550.120.0
2025-04-15 19:55:23.642969: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:249] kernel version seems to match DSO: 550.120.0


In [13]:
# Compiling the model
binary_model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

In [14]:
# Fit the model
binary_model.fit(
    binary_train_generator,
    validation_data=binary_val_generator,
    epochs=6,
    steps_per_epoch=int(np.ceil(len(train_df) / BATCH_SIZE)),
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)
    ],
    verbose=1
)

/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/6
 466/1019 ━━━━━━━━━━━━━━━━━━━━ 7:29 812ms/step - accuracy: 0.9180 - loss: 0.3351 - precision: 0.9392 - recall: 0.9764

/home/shadybea/anaconda3/envs/dl/lib/python3.11/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (115600000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


1019/1019 ━━━━━━━━━━━━━━━━━━━━ 925s 881ms/step - accuracy: 0.9229 - loss: 0.3098 - precision: 0.9345 - recall: 0.9870 - val_accuracy: 0.9200 - val_loss: 0.2807 - val_precision: 0.9200 - val_recall: 1.0000 - learning_rate: 0.0100
Epoch 2/6
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 893s 876ms/step - accuracy: 0.9252 - loss: 0.2643 - precision: 0.9261 - recall: 0.9990 - val_accuracy: 0.9200 - val_loss: 0.4148 - val_precision: 0.9200 - val_recall: 1.0000 - learning_rate: 0.0100
Epoch 3/6
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 0s 812ms/step - accuracy: 0.9352 - loss: 0.2338 - precision: 0.9361 - recall: 0.9990
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 894s 877ms/step - accuracy: 0.9352 - loss: 0.2338 - precision: 0.9361 - recall: 0.9990 - val_accuracy: 0.9200 - val_loss: 0.2939 - val_precision: 0.9200 - val_recall: 1.0000 - learning_rate: 0.0100
Epoch 4/6
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 895s 878ms/step - accuracy: 0.9319 - loss: 0.2272 - precision

In [17]:
# Export the model
binary_model.save('../models/binary_model.keras')

In [ ]:
# Load the model
binary_model = load_model('../models/binary_model.keras')

In [15]:
# Make predictions
predictions = binary_model.predict(
    binary_test_generator
    ,steps=len(binary_test_generator)
    ,verbose=1
)

300/300 ━━━━━━━━━━━━━━━━━━━━ 113s 370ms/step


In [16]:
# Get the true labels
y_true = binary_test_generator.labels

# Convert predictions to class labels
y_pred = (predictions > 0.5).astype(int).flatten()

# Calculate metrics
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall: {recall_score(y_true, y_pred):.4f}")

Accuracy: 0.9328
Precision: 0.9328
Recall: 1.0000


In [39]:
# Get unique values and their counts
unique, counts = np.unique(y_pred, return_counts=True)

# Display the result
value_counts = dict(zip(unique, counts))
print(value_counts)

{1: 2397}


In [ ]:
# Save the predictions
file_paths = binary_test_generator.filenames
pred_df = pd.DataFrame({
    'file_path': file_paths,
    'predictions': (predictions > 0.5).astype(int).flatten()
})

# Export to csv
# pred_df.to_csv('../models/binary_predictions.csv', index=False)